<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Median, MAD and IQR

**[Median, MAD and IQR](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/)** · Anomaly Detection: Find What Is Unusual · Module 2, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build a centre and a spread that one extreme value does not move: the median and the MAD, and the boxplot rule with the IQR. Then you give every customer a baseline of their own, and add the rows to the comparison table.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | The z-score, from [Z-scores, and why the mean is fooled](https://learning.nextia-ai.com/courses/anomaly-detection/m02/z-scores-and-why-the-mean-is-fooled/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M02-L02-median-mad-and-iqr/median-mad-and-iqr-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A centre that one value cannot move: the median and the MAD

In the last lesson, one 900-dollar payment moved the mean from 14.14 to 124.88 dollars. The 60-dollar payment then looked normal.

The **median** is the middle value when you sort the values. Half of the values are below it, half above. To move the median, you must move half of the values, not one.

The **MAD** (median absolute deviation) is the spread made the same way. Take the distance of each value from the median. The MAD is the median of these distances.

The **robust z-score** uses them in place of the mean and the standard deviation:

robust z = (value − median) ÷ (1.4826 × MAD)

The number 1.4826 makes the MAD the same size as a standard deviation when the data has the normal bell shape. So a robust z-score of 3 means about the same as a z-score of 3.

> **Check.** You should see `median 14.00, MAD 1.00, robust z-score of 60 dollars: 31.0`.

In [ ]:
usual = [12.0, 15.0, 14.0, 13.0, 16.0, 15.0, 14.0]     # the same seven payments as in the last lesson
new_payment = 60.0

def robust_z(x, values):
    """The median, the MAD and the robust z-score of x against the values."""
    values = np.array(values)
    median = np.median(values)
    mad = np.median(np.abs(values - median))      # the median distance from the median
    return median, mad, (x - median) / (1.4826 * mad)

median, mad, rz = robust_z(new_payment, usual)
print("distances from the median:", sorted(np.abs(np.array(usual) - median).tolist()))
print(f"median {median:.2f}, MAD {mad:.2f}, robust z-score of 60 dollars: {rz:.1f}")

> **Predict.** The 900-dollar payment joins the history. How much does the median move? And the MAD?

In [ ]:
with_huge = usual + [900.0]                            # one huge payment joins the history
median2, mad2, rz2 = robust_z(new_payment, with_huge)
print(f"median {median2:.2f}, MAD {mad2:.2f}, robust z-score of 60 dollars: {rz2:.1f}")

> **Check.** You should see `median 14.50, MAD 1.00, robust z-score of 60 dollars: 30.7`.

The median moved by 0.50 dollars and the MAD did not move. The 60-dollar payment is still very far from usual. Methods like this are called **robust**: one extreme value does not change them much.

## 2. The boxplot rule: Q1, Q3 and the IQR

Another robust spread uses quartiles. **Q1** (the first quartile) has a quarter of the values below it; **Q3** has three quarters below it. The **IQR** (interquartile range) is Q3 − Q1: the width of the middle half of the values.

The **boxplot rule** calls a value an outlier when it is more than 1.5 IQR above Q3 (or below Q1). The limit Q3 + 1.5 × IQR is the **upper fence**.

> **Check.** You should see an upper fence of 17.50, and both 900.0 and 60.0 above it.

In [ ]:
q1, q3 = np.percentile(with_huge, [25, 75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
print(f"Q1 {q1:.2f}, Q3 {q3:.2f}, IQR {iqr:.2f}, upper fence {upper_fence:.2f}")
print("above the fence:", [v for v in with_huge + [new_payment] if v > upper_fence])

## 3. One baseline for the whole shop

Now the real payments. The detector learns the median, the MAD and the quartiles from the **reference months** (June to August), and scores the **evaluation months** (September to November).

> **Check.** You should see a median of 44.78, a MAD of 25.02 and an upper fence of 218.54.

In [ ]:
import compare
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
g_median = reference.amount.median()
g_mad = (reference.amount - g_median).abs().median()
g_q1, g_q3 = reference.amount.quantile([0.25, 0.75])
print(f"reference months: median {g_median:.2f}, MAD {g_mad:.2f}, Q1 {g_q1:.2f}, Q3 {g_q3:.2f}, "
      f"upper fence {g_q3 + 1.5 * (g_q3 - g_q1):.2f}")

> **Predict.** The robust z-score is not fooled by one huge payment. Will it find more planted frauds than the z-score of the last lesson (precision at budget 0.003)?

In [ ]:
table = compare.Table()
rz_global = ((evaluation.amount - g_median) / (1.4826 * g_mad)).abs()
table.add(compare.score_payments(evaluation, rz_global, "MAD (global)", "|robust z| of the amount, one baseline",
                                 explain_minutes=1))
iqr_flag = (evaluation.amount > g_q3 + 1.5 * (g_q3 - g_q1)).astype(int)
table.add(compare.score_payments(evaluation, iqr_flag, "IQR rule (global)", "amount above Q3 + 1.5 IQR",
                                 threshold=1, explain_minutes=1))
table.show()
flagged = evaluation[iqr_flag == 1]
print(f"the IQR rule flags {len(flagged):,} payments; {flagged.account_type.eq('business').mean():.1%} are business payments")

> **Check.** You should see a precision at budget of 0.003 for the global MAD and 0.006 for the IQR rule, with 236.2 IQR alerts a day.

No better. With one baseline for everybody, the most unusual amounts are still the largest ones: business orders. Robust statistics fix the effect of an extreme value. They do not fix a wrong question. The question must be "unusual **for this customer**".

## 4. A baseline per group, then per customer

Business customers pay much more than personal customers. Run the next cell to see each group's own baseline.

In [ ]:
by_type = reference.groupby("account_type").amount.agg(
    ["median", lambda s: (s - s.median()).abs().median(), "size"])
by_type.columns = ["median", "mad", "payments"]
print(by_type.round(2).to_string())

In [ ]:
fig, ax = plt.subplots()
ax.boxplot([reference.amount[reference.account_type == t] for t in ["personal", "business"]],
           orientation="horizontal", tick_labels=["personal", "business"], whis=1.5, flierprops={"markersize": 2})
ax.set_xscale("log")
ax.set_xlabel("amount in the reference months, dollars (log scale)")
plt.show()

The middle half of the business payments is far to the right of the personal ones. A 300-dollar payment is ordinary for a business and unusual for a person.

Now each customer gets a median and a MAD from their own reference payments. A customer with fewer than 5 reference payments uses the baseline of their account type.

> **Predict.** Write down your guess for the precision at budget of MAD per customer. The z-score per customer of the last lesson had 0.045.

In [ ]:
g = reference.groupby("customer_id").amount
cust = pd.DataFrame({"n": g.size(), "median": g.median(),
                     "mad": g.apply(lambda s: (s - s.median()).abs().median())})
known = evaluation.customer_id.map(cust["n"]) >= 5               # at least 5 payments to learn from
c_median = evaluation.customer_id.map(cust["median"]).where(known).fillna(evaluation.account_type.map(by_type["median"]))
c_mad = evaluation.customer_id.map(cust["mad"]).where(known).fillna(evaluation.account_type.map(by_type["mad"]))
c_mad = np.maximum(c_mad, 1.0)                                 # a floor of 1 dollar for the spread (section 5)
rz_customer = ((evaluation.amount - c_median) / (1.4826 * c_mad)).abs()
row = table.add(compare.score_payments(evaluation, rz_customer, "MAD per customer",
                                       "|robust z| against the customer's own median", explain_minutes=1))
table.show()
print(f"customers with a baseline: {int((cust['n'] >= 5).sum()):,}; "
      f"evaluation payments that use the account type's baseline: {int((~known).sum()):,}")

> **Check.** You should see a precision at budget of 0.048 and a recall of 0.078 for MAD per customer.

About 16 times better than one baseline for everybody, and a little better than the z-score per customer. Still, only about 2 or 3 of the 50 daily alerts are planted frauds. The next lesson shows why: many frauds are not about the amount.

### The mean fooled, in the real data

Here is one customer whose history has one large payment. Run the next cell.

In [ ]:
history = reference[reference.customer_id == "C-06547"].amount
odd = evaluation[evaluation.payment_id == "P-0199876"].iloc[0]
print("reference payments:", history.round(2).tolist())
print(f"new payment: {odd.amount:.2f} dollars on {odd.ts}")
print(f"mean {history.mean():.2f}, standard deviation {history.std():.2f}, "
      f"z-score {(odd.amount - history.mean()) / history.std():.2f}")
h_median, h_mad, h_rz = robust_z(odd.amount, history)
print(f"median {h_median:.2f}, MAD {h_mad:.2f}, robust z-score {h_rz:.2f}")

One payment of 248.87 dollars made the standard deviation large, so the z-score of the new payment is only -1.46. The median and the MAD ignore it: the robust z-score is -8.51. This payment is a planted fraud: a tiny payment that tests whether a stolen card works.

## 5. A common mistake: a MAD of zero

If more than half of a customer's payments have exactly the same amount, the MAD is 0. Then the robust z-score divides by zero. Run the next cell.

In [ ]:
same = [20.0, 20.0, 20.0, 25.0, 20.0]                   # four of five payments are exactly 20 dollars
median0, mad0, rz0 = robust_z(21.0, same)
print(f"median {median0:.2f}, MAD {mad0:.2f}, robust z-score of 21 dollars: {rz0}")
print("customers with a baseline and a MAD below 1 dollar:", int(((cust["n"] >= 5) & (cust["mad"] < 1)).sum()))

NumPy prints a `RuntimeWarning`, and you get `inf` (infinity) for a payment only 1 dollar above usual. An `inf` score sorts first and takes a place in the budget. A very small MAD has the same effect, only weaker: in the data, 66 customers with a baseline have a MAD below 1 dollar. The fix is a floor for the spread, as in section 4. Run the next cell.

In [ ]:
rz0_fixed = (21.0 - median0) / (1.4826 * max(mad0, 1.0))   # a floor of 1 dollar for the spread
print(f"robust z-score of 21 dollars, with a floor: {rz0_fixed:.2f}")

## 6. Check yourself

**Your turn 1.** In section 1, what is the robust z-score of the 60-dollar payment **after** the huge payment? Write it as `robust_after` (with 1 decimal).

In [ ]:
robust_after = ...   # your answer, a number

In [ ]:
expect('robust_after', robust_after, 30.7, tolerance=0.01)

**Your turn 2.** Compute the upper fence of the boxplot rule for **personal** accounts only, in the reference months. Write it as `personal_fence`, rounded to 2 decimals.

In [ ]:
personal_fence = ...   # your code

In [ ]:
expect('personal_fence', personal_fence, 102.81, tolerance=0.0001)

**Your turn 3.** Score only the high side of MAD per customer: keep the sign of `(evaluation.amount - c_median) / (1.4826 * c_mad)`, so that amounts below the usual one get low scores. Keep the row as `up_row`. Is it better than the size of the score?

In [ ]:
up_row = ...   # your code: compare.score_payments(evaluation, ..., "MAD per customer, high side", "only amounts above usual")

In [ ]:
expect_hash('the precision of the high-side MAD score', up_row['precision at budget'] if isinstance(up_row, dict) else None, 'e0313c5be39e9622')

The high side gives 0.033, less than 0.048. Some planted frauds are far **below** the usual amount: tiny payments that test a card, as in section 4. Keep both sides.

## Recap

- The median and the MAD do not move when one extreme value joins: the robust z-score of 60 dollars went from 31.0 to 30.7.
- The boxplot rule flags values above Q3 + 1.5 × IQR.
- One robust baseline for the whole shop still finds almost nothing (0.003). One per customer finds more (0.048).
- A MAD can be 0: give the spread a floor.

The next lesson writes rules for the shape of a fraud, and compares them with these scores.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Rules, and when a rule is the right answer](https://learning.nextia-ai.com/courses/anomaly-detection/m02/rules-and-when-a-rule-is-the-right-answer/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.